# AEGIS-MD — full pipeline in one notebook (GPU T4 + Internet ON)
Sections 1–5 = the five notebooks. Run cells in order; every section saves its outputs to /kaggle/working and the next section reloads them.


In [ ]:
%%writefile aegis_core.py
# =====================================================================
# aegis_core.py  —  AEGIS-MD core library
# Adaptive Evidence-Gap Investigation of Safety signals for Medical Devices
#
# Shared by all notebooks (each notebook writes this file to disk first,
# so every notebook is self-contained on Kaggle).
# =====================================================================
import os, re, glob, json, math, time, warnings
from collections import Counter, defaultdict
import numpy as np
import pandas as pd
from scipy import stats, sparse

warnings.filterwarnings("ignore")

# ---------------------------------------------------------------------
# Configuration (change here, or override in a notebook before use)
# ---------------------------------------------------------------------
CFG = dict(
    DATA_START="2014-01",   # first month of MAUDE data pulled
    DATA_END="2025-12",     # last month of MAUDE data pulled
    PANEL_START="2016-01",  # first evaluation month t
    PANEL_END="2025-06",    # last evaluation month t (needs HORIZON months of labels after it)
    TRAIN_END="2020-12",    # t <= TRAIN_END  -> train
    VAL_END="2022-06",      # TRAIN_END < t <= VAL_END -> validation ; later -> test
    HORIZON=6,              # predict a new recall initiated in (t, t+HORIZON]
    WASHOUT=6,              # drop t if a recall of that code was initiated in (t-WASHOUT, t]
    RECENT=3,               # recent window = months t-2..t
    BASELINE=12,            # baseline window = 12 months before the recent window
    LABEL_CLASSES=["Class I"],   # ["Class I"] (primary) or ["Class I","Class II"]
    K_PER_GAP=10,           # documents read per evidence-gap retrieval
    SEED=42,
)

DEEP_SOURCES = ["mechanism", "investigation", "batch", "precedent"]
TIER0_SOURCES = ["statistical", "temporal", "ml_anomaly", "severity", "problem_shift"]

# Gap specifications: what each evidence gap needs, and how it is queried.
GAP_SPECS = {
    "mechanism": dict(
        question="Is there a consistent, specific failure mechanism behind the recent reports?",
        proto="The device failed or malfunctioned in a specific, repeatable way; the report describes the failure mechanism and the component involved.",
        terms=["failure", "failed", "malfunction", "broke", "crack", "leak", "error", "alarm", "component"],
        use_emerging=True),
    "investigation": dict(
        question="What did the manufacturer's own investigation of returned devices conclude?",
        proto="The manufacturer evaluated the returned device; the investigation confirmed the reported failure and identified the root cause.",
        terms=["evaluation", "evaluated", "investigation", "returned", "analysis", "confirmed", "root", "cause", "found", "reproduced"],
        use_emerging=False),
    "batch": dict(
        question="Are the affected units concentrated in specific lots, batches or models?",
        proto="Multiple units from the same lot or manufacturing batch were affected by the same manufacturing defect.",
        terms=["lot", "batch", "manufacturing", "units", "same", "defect", "production", "nonconformance", "supplier"],
        use_emerging=False),
    "precedent": dict(
        question="Does this failure resemble defects that previously led to recalls?",
        proto="The failure resembles a known product defect that previously led to a corrective action or product recall.",
        terms=["defect", "design", "manufacturing", "failure", "component"],
        use_emerging=True),
}
GENERIC_QUERY = dict(
    proto="A report of an adverse event or problem involving the medical device.",
    terms=["adverse", "event", "problem", "malfunction", "patient", "device"])

# ---------------------------------------------------------------------
# Month helpers  (month index mi = year*12 + month-1)
# ---------------------------------------------------------------------
def ym(s):
    y, m = map(int, str(s)[:7].split("-")); return y * 12 + m - 1

def ym_str(mi):
    mi = int(mi); return f"{mi // 12}-{mi % 12 + 1:02d}"

def to_mi(series):
    d = pd.to_datetime(series, errors="coerce", format="mixed") if hasattr(pd, "to_datetime") else series
    return (d.dt.year * 12 + d.dt.month - 1).astype("float")

def mi_from_yyyymmdd(s):
    s = str(s or "")
    s = s.replace("-", "")
    if len(s) < 6 or not s[:6].isdigit():
        return np.nan
    return int(s[:4]) * 12 + int(s[4:6]) - 1

def split_of(mi, cfg=CFG):
    if mi <= ym(cfg["TRAIN_END"]): return "train"
    if mi <= ym(cfg["VAL_END"]): return "val"
    return "test"

# ---------------------------------------------------------------------
# File discovery (Kaggle: /kaggle/working and attached inputs)
# ---------------------------------------------------------------------
def data_roots():
    roots = []
    if os.environ.get("AEGIS_DIR"): roots.append(os.environ["AEGIS_DIR"])
    roots += ["/kaggle/working", "/kaggle/input", "."]
    return [r for r in roots if os.path.isdir(r)]

def find_file(name, required=True):
    for r in data_roots():
        p = os.path.join(r, name)
        if os.path.exists(p): return p
        hits = glob.glob(os.path.join(r, "**", name), recursive=True)
        if hits: return sorted(hits, key=len)[0]
    if required:
        raise FileNotFoundError(f"{name} not found. Attach the previous notebook's output as an input (Add Input -> Notebook Output).")
    return None

def out_dir():
    d = os.environ.get("AEGIS_OUT") or ("/kaggle/working" if os.path.isdir("/kaggle/working") else ".")
    os.makedirs(d, exist_ok=True); return d

# ---------------------------------------------------------------------
# Text utilities
# ---------------------------------------------------------------------
STOP = set("""a an the and or of to in on for with without by from at as is are was were be been being this that these those it its
into than then there their they them he she his her patient patients device devices reported report reports reportedly per also
not no yes any all some such which who whom what when where while during after before about over under between via
had has have having did does do done will would could should may might can cannot unable however therefore further
information additional event events date dates time times use used using user customer facility hospital
mfr manufacturer company firm product products model lot serial number numbers unknown na n/a""".split())
TOKEN_RE = re.compile(r"[a-z][a-z0-9\-]{2,}")

def tokenize(s, keep_stop=False):
    toks = TOKEN_RE.findall(str(s or "").lower())
    return toks if keep_stop else [t for t in toks if t not in STOP]

# Leakage guard: MAUDE narratives can be updated by supplemental reports
# after a recall. Every sentence that mentions a recall / field action is removed.
RECALL_MENTION = re.compile(
    r"(recall|recalled|field safety|field action|field correction|correction and removal|"
    r"\bfsca\b|\bfsn\b|z-\d{3,4}-\d{4}|urgent medical device|advisory notice|removal notice)", re.I)
SENT_SPLIT = re.compile(r"(?<=[\.\!\?;])\s+")

def scrub_recall_mentions(text):
    if not text: return "", 0
    sents = SENT_SPLIT.split(str(text))
    keep = [s for s in sents if not RECALL_MENTION.search(s)]
    return " ".join(keep), len(sents) - len(keep)

# Regex stance (CPU fallback when no NLI model is available)
CONFIRM_RE = re.compile(
    r"(was|were|has been|have been|is) confirmed|confirmed the (reported|complaint|failure|issue|allegation)|"
    r"root cause (was|is|has been) (determined|identified|attributed|found)|"
    r"(failure|problem|issue|defect) (was|is|were) (found|observed|reproduced|duplicated|verified)|"
    r"(found|observed) to be (cracked|broken|defective|damaged|leaking|fractured|separated|occluded)|"
    r"(evaluation|analysis|investigation|inspection) (found|revealed|identified|determined|showed)", re.I)
REFUTE_RE = re.compile(
    r"could not be (confirmed|reproduced|duplicated|determined|verified)|"
    r"no (device )?(malfunction|anomal\w*|defect|problem|issue|fault|failure)s? (was|were )?(found|identified|observed|detected|noted)|"
    r"(was|were|has not been|have not been) not (been )?returned|not returned for (evaluation|analysis)|unable to (confirm|reproduce|duplicate|verify)|"
    r"functioned (as intended|properly|normally|as designed)|(met|meets|within) (all )?(specification|specifications)|"
    r"\b(user|use) error\b|not confirmed|unconfirmed|no evidence of (a )?(device )?(malfunction|defect)", re.I)

def regex_stance(text):
    t = str(text or "")
    if not t.strip(): return np.nan, np.nan
    c = 0.85 if CONFIRM_RE.search(t) else 0.10
    r = 0.85 if REFUTE_RE.search(t) else 0.10
    return c, r

# ---------------------------------------------------------------------
# Sparse BM25 index over all records (vectorised; fast per-case scoring)
# ---------------------------------------------------------------------
class BM25Matrix:
    def __init__(self, texts, k1=1.2, b=0.75, min_df=3, max_features=60000):
        from sklearn.feature_extraction.text import CountVectorizer
        self.cv = CountVectorizer(tokenizer=tokenize, lowercase=False, token_pattern=None,
                                  min_df=min_df, max_features=max_features)
        tf = self.cv.fit_transform(texts).tocsr().astype(np.float32)
        N = tf.shape[0]
        df = np.diff(tf.tocsc().indptr)
        idf = np.log(1 + (N - df + 0.5) / (df + 0.5)).astype(np.float32)
        dl = np.asarray(tf.sum(1)).ravel()
        avgdl = max(dl.mean(), 1.0)
        tf = tf.tocoo()
        denom = tf.data + k1 * (1 - b + b * dl[tf.row] / avgdl)
        w = idf[tf.col] * tf.data * (k1 + 1) / denom
        self.W = sparse.csr_matrix((w, (tf.row, tf.col)), shape=tf.shape)
        self.vocab = self.cv.vocabulary_

    def qvec(self, tokens):
        idx = sorted({self.vocab[t] for t in tokens if t in self.vocab})
        v = np.zeros(self.W.shape[1], dtype=np.float32); v[idx] = 1.0
        return v

    def score(self, rows, qv):
        return np.asarray(self.W[rows] @ qv).ravel()

def _minmax(x):
    x = np.asarray(x, dtype=np.float32)
    lo, hi = (x.min(), x.max()) if len(x) else (0, 0)
    return (x - lo) / (hi - lo) if hi > lo else np.zeros_like(x)

def hybrid_retrieve(rows, bm25, emb, q_tokens, proto_vec, k, alpha=0.5):
    """Hybrid sparse (BM25) + dense (embedding) retrieval restricted to candidate rows."""
    rows = np.asarray(rows)
    if len(rows) <= k: return rows
    s_bm = _minmax(bm25.score(rows, bm25.qvec(q_tokens)))
    s_dn = _minmax(emb[rows].astype(np.float32) @ proto_vec.astype(np.float32))
    s = alpha * s_bm + (1 - alpha) * s_dn
    return rows[np.argsort(-s, kind="stable")[:k]]

# ---------------------------------------------------------------------
# Tier-0 (routine surveillance) detectors
# ---------------------------------------------------------------------
def mann_kendall(x):
    x = np.asarray(x, dtype=float); n = len(x)
    if n < 4 or np.all(x == x[0]): return 0.0, 0.0, 1.0
    s = 0.0
    for i in range(n - 1): s += np.sign(x[i + 1:] - x[i]).sum()
    _, counts = np.unique(x, return_counts=True)
    var = (n * (n - 1) * (2 * n + 5) - np.sum(counts * (counts - 1) * (2 * counts + 5))) / 18.0
    z = 0.0 if var <= 0 else ((s - 1) / math.sqrt(var) if s > 0 else ((s + 1) / math.sqrt(var) if s < 0 else 0.0))
    tau = s / (0.5 * n * (n - 1))
    p = 2 * (1 - stats.norm.cdf(abs(z)))
    return tau, z, p

def sen_slope(x):
    x = np.asarray(x, dtype=float); n = len(x)
    if n < 3: return 0.0
    sl = [(x[j] - x[i]) / (j - i) for i in range(n - 1) for j in range(i + 1, n)]
    return float(np.median(sl))

def jsd(p_counts, q_counts):
    keys = set(p_counts) | set(q_counts)
    if not keys: return 0.0
    p = np.array([p_counts.get(k, 0) for k in keys], float) + 0.5
    q = np.array([q_counts.get(k, 0) for k in keys], float) + 0.5
    p /= p.sum(); q /= q.sum(); m = 0.5 * (p + q)
    return float(0.5 * np.sum(p * np.log(p / m)) + 0.5 * np.sum(q * np.log(q / m)))

def tier0_for_code(n_by_month, months, agg, cfg=CFG):
    """Tier-0 evidence for every month of one product code.
    n_by_month: dict mi->total reports (openFDA counts); agg: dict mi->dict of sample aggregates."""
    R, B = cfg["RECENT"], cfg["BASELINE"]
    rows, cusum = [], 0.0
    for t in months:
        rec = [n_by_month.get(m, 0) for m in range(t - R + 1, t + 1)]
        base = [n_by_month.get(m, 0) for m in range(t - R - B + 1, t - R + 1)]
        mu = max(np.mean(base), 0.25)
        n_rec = sum(rec)
        p_pois = stats.poisson.sf(n_rec - 1, mu * R)
        last12 = [n_by_month.get(m, 0) for m in range(t - 11, t + 1)]
        tau, z_mk, p_mk = mann_kendall(last12)
        slope = sen_slope(last12) / (np.mean(last12) + 1.0)
        persist = sum(stats.poisson.sf(n_by_month.get(m, 0) - 1, mu) < 0.05 for m in range(t - 5, t + 1))
        n_t = n_by_month.get(t, 0)
        cusum = max(0.0, cusum + (n_t - mu) / math.sqrt(mu + 1.0) - 0.5)
        # sample-based structured aggregates (severity, problem-code mix, reporting reliability)
        def pool(ms):
            out = dict(n=0, serious=0, mfr_src=0, text=0, prob=Counter())
            for m in ms:
                a = agg.get(m)
                if a is None: continue
                out["n"] += a["n"]; out["serious"] += a["serious"]; out["mfr_src"] += a["mfr_src"]
                out["text"] += a["text"]; out["prob"].update(a["prob"])
            return out
        w = pool(range(t - R + 1, t + 1)); b = pool(range(t - R - B + 1, t - R + 1))
        p1 = (w["serious"] + 0.5) / (w["n"] + 1); p0 = (b["serious"] + 0.5) / (b["n"] + 1)
        pp = (w["serious"] + b["serious"] + 1) / (w["n"] + b["n"] + 2)
        se = math.sqrt(pp * (1 - pp) * (1 / (w["n"] + 1) + 1 / (b["n"] + 1)))
        rows.append(dict(
            mi=t, n_recent=n_rec, base_mu=mu,
            pois_score=min(-math.log10(max(p_pois, 1e-12)), 12.0),
            log_ratio=math.log((n_rec + 0.5) / (mu * R + 0.5)),
            mk_z=z_mk, mk_tau=tau, sen_norm=slope, persist=persist, cusum=cusum,
            sev_z=(p1 - p0) / se if se > 0 else 0.0, sev_share=p1,
            prob_jsd=jsd(w["prob"], b["prob"]),
            reliab=(w["mfr_src"] + 1) / (w["n"] + 2), completeness=(w["text"] + 1) / (w["n"] + 2),
            n_sample_recent=w["n"]))
    return pd.DataFrame(rows)

TIER0_GROUPS = {
    "statistical": ["pois_score", "log_ratio", "cusum"],
    "temporal": ["mk_z", "sen_norm", "persist"],
    "ml_anomaly": ["if_score"],
    "severity": ["sev_z", "sev_share"],
    "problem_shift": ["prob_jsd"],
}

def emerging_terms(win_prob, base_prob, topn=3):
    tw = sum(win_prob.values()) + 1; tb = sum(base_prob.values()) + 1
    sc = {k: math.log((v + 1) / tw) - math.log((base_prob.get(k, 0) + 1) / tb) for k, v in win_prob.items()}
    top = [k for k, _ in sorted(sc.items(), key=lambda kv: -kv[1])[:topn]]
    toks = []
    for k in top: toks += tokenize(k)
    return top, toks

# ---------------------------------------------------------------------
# Deep evidence extracted from a set of retrieved documents S
# ---------------------------------------------------------------------
DEEP_FEATURES = {
    "mechanism": ["mech_coh", "mech_shift"],
    "investigation": ["inv_confirm", "inv_refute", "inv_cov"],
    "batch": ["lot_hhi", "model_hhi", "lot_cov"],
    "precedent": ["prec_top", "prec_max"],
}

def _hhi(values):
    v = [x for x in values if x]
    if len(v) < 2: return 0.0
    c = np.array(list(Counter(v).values()), float); s = c / c.sum()
    return float((s ** 2).sum())

def deep_evidence(S, R, ctx, sources=DEEP_SOURCES):
    """S: array of record row-ids; R: dict of record arrays; ctx: case context (baseline centroid etc.).
    Returns dict feature->value plus per-source reliability rho_<source>."""
    out = {}
    S = np.asarray(S, dtype=int)
    if len(S) == 0:
        for s in sources:
            for f in DEEP_FEATURES[s]: out[f] = np.nan
            out["rho_" + s] = 0.0
        return out
    rho = 0.5 + 0.5 * float(np.mean(R["has_text"][S] & R["mand_src"][S]))
    if "mechanism" in sources:
        E = R["emb"][S].astype(np.float32)
        c = E.mean(0); cn = c / (np.linalg.norm(c) + 1e-9)
        out["mech_coh"] = float((E @ cn).mean()) - ctx["base_coh"]
        out["mech_shift"] = float(1 - cn @ ctx["base_cen"]) if ctx["base_cen"] is not None else 0.0
        out["rho_mechanism"] = rho
    if "investigation" in sources:
        has = R["has_mfr"][S]
        if has.any():
            out["inv_confirm"] = float(np.nanmean(R["p_confirm"][S][has]))
            out["inv_refute"] = float(np.nanmean(R["p_refute"][S][has]))
        else:
            out["inv_confirm"] = out["inv_refute"] = np.nan
        out["inv_cov"] = float(has.mean())
        out["rho_investigation"] = rho
    if "batch" in sources:
        lots = R["lot"][S]; models = R["model"][S]
        out["lot_hhi"] = _hhi(lots); out["model_hhi"] = _hhi(models)
        out["lot_cov"] = float(np.mean([bool(x) for x in lots]))
        out["rho_batch"] = rho
    if "precedent" in sources:
        ps = np.sort(R["prec_sim"][S])[::-1]
        ps = ps[np.isfinite(ps)]
        out["prec_top"] = float(ps[:3].mean()) if len(ps) else np.nan
        out["prec_max"] = float(ps[0]) if len(ps) else np.nan
        out["rho_precedent"] = rho
    return out

LEVEL_FEATURES = {"mech_coh", "inv_confirm", "inv_refute", "lot_hhi", "model_hhi", "prec_top", "prec_max"}

def contrastive_gap_evidence(S_w, S_b, R, ctx, source, rate=None):
    """Query-conditioned contrastive evidence: the same gap query is run on the recent window
    (S_w) and on the code's own baseline window (S_b); level features are reported as
    window-minus-baseline differences so that query-induced selection bias cancels out."""
    fw = deep_evidence(S_w, R, ctx, sources=[source])
    fb = deep_evidence(S_b, R, ctx, sources=[source]) if len(S_b) else None
    out = {}
    for f in DEEP_FEATURES[source]:
        v = fw.get(f, np.nan)
        if f in LEVEL_FEATURES and fb is not None and np.isfinite(fb.get(f, np.nan)) and np.isfinite(v):
            v = v - fb[f]
        out[f] = v
    if source == "mechanism" and len(S_w) and len(S_b):
        cw = R["emb"][S_w].astype(np.float32).mean(0); cb = R["emb"][S_b].astype(np.float32).mean(0)
        out["mech_shift"] = float(1 - (cw @ cb) / (np.linalg.norm(cw) * np.linalg.norm(cb) + 1e-9))
    out["rate_" + source] = rate if rate is not None else np.nan
    out["rho_" + source] = fw.get("rho_" + source, 0.0)
    return out

def query_rate_shift(win, base, emb, proto, thr):
    """Change in the share of reports that match a gap query (dense similarity above a fixed
    corpus-level threshold) between the recent and the baseline window. No reading needed."""
    if len(win) == 0: return np.nan
    hw = float((emb[win].astype(np.float32) @ proto >= thr).sum())
    hb = float((emb[base].astype(np.float32) @ proto >= thr).sum()) if len(base) else 0.0
    return math.log((hw + 0.5) / (len(win) + 1)) - math.log((hb + 0.5) / (len(base) + 1))

CONTRAST_FEATURES = {s: DEEP_FEATURES[s] + ["rate_" + s] for s in DEEP_FEATURES}

# ---------------------------------------------------------------------
# Reliability-discounted Bayesian evidence fusion + Value-of-Information
# ---------------------------------------------------------------------
def _logit(p): p = np.clip(p, 1e-6, 1 - 1e-6); return np.log(p / (1 - p))
def _sigmoid(z): return 1 / (1 + np.exp(-z))

class SourceCalibrator:
    """Maps one evidence source's raw features to a log-likelihood ratio (LLR)
    relative to the base rate: LLR = logit(P(y|x)) - logit(a)."""
    def __init__(self, feats, C=0.5):
        self.feats, self.C = feats, C
    def _prep(self, X):
        X = X[self.feats].astype(float).copy()
        miss = X.isna().all(axis=1).values
        X = X.fillna(self.med)
        return X.values, miss
    def fit(self, X, y):
        from sklearn.linear_model import LogisticRegression
        from sklearn.preprocessing import StandardScaler
        self.med = X[self.feats].median()
        self.med = self.med.fillna(0.0)
        Xv, miss = self._prep(X)
        self.sc = StandardScaler().fit(Xv[~miss])
        self.lr = LogisticRegression(C=self.C, max_iter=2000).fit(self.sc.transform(Xv[~miss]), y[~miss])
        self.la = _logit(np.clip(y.mean(), 1e-4, 1 - 1e-4))
        return self
    def llr(self, X):
        Xv, miss = self._prep(X)
        p = self.lr.predict_proba(self.sc.transform(Xv))[:, 1]
        l = _logit(p) - self.la
        l[miss] = 0.0          # absent evidence carries no information
        return np.clip(l, -6.0, 6.0)   # bound any single source's influence

class EvidenceFusion:
    """z = b0 + sum_k w_k * rho_k * LLR_k   (reliability-discounted log-odds pooling).
    Calibrators are fit on TRAIN; pooling weights on VALIDATION; VoI uses out-of-fold
    class-conditional LLR distributions from TRAIN."""
    def __init__(self, groups, use_rho=True, C=0.5):
        self.groups, self.use_rho, self.C = groups, use_rho, C
    def _L(self, X, rho=None):
        L = np.column_stack([self.cal[s].llr(X) for s in self.groups])
        if self.use_rho and rho is not None:
            L = L * rho
        return L
    def _rho(self, X):
        return np.column_stack([X["rho_" + s].fillna(1.0).values if ("rho_" + s) in X else np.ones(len(X))
                                for s in self.groups])
    def fit(self, Xtr, ytr, Xva, yva, groups_tr=None):
        from sklearn.linear_model import LogisticRegression
        from sklearn.model_selection import GroupKFold
        self.cal = {s: SourceCalibrator(f, self.C).fit(Xtr, ytr) for s, f in self.groups.items()}
        Lva = self._L(Xva, self._rho(Xva))
        self.stack = LogisticRegression(C=1.0, max_iter=2000).fit(Lva, yva)
        self.w = self.stack.coef_.ravel(); self.b0 = float(self.stack.intercept_[0])
        # out-of-fold LLR samples (train) for VoI predictive distributions
        oof = np.zeros((len(Xtr), len(self.groups)))
        g = groups_tr if groups_tr is not None else np.arange(len(Xtr)) % 5
        for tr, te in GroupKFold(n_splits=5).split(Xtr, ytr, g):
            for j, (s, f) in enumerate(self.groups.items()):
                c = SourceCalibrator(f, self.C).fit(Xtr.iloc[tr], ytr[tr])
                oof[te, j] = c.llr(Xtr.iloc[te])
        oof = oof * (self._rho(Xtr) if self.use_rho else 1.0)
        rng = np.random.default_rng(0)
        self.samples = {}
        for j, s in enumerate(self.groups):
            for yv in (0, 1):
                v = oof[ytr == yv, j]; v = v[v != 0] if (v != 0).any() else v
                self.samples[(s, yv)] = rng.choice(v, size=min(400, len(v)), replace=len(v) < 400) if len(v) else np.zeros(1)
        return self
    def contributions(self, X):
        return self._L(X, self._rho(X)) * self.w
    def predict_logit(self, X, mask=None):
        C = self.contributions(X)
        if mask is not None: C = C * mask
        return self.b0 + C.sum(1)
    def predict(self, X, mask=None):
        return _sigmoid(self.predict_logit(X, mask))

def bayes_risk(p, tau):
    return np.minimum(tau * (1 - p), (1 - tau) * p)

def voi(z, j_name, fusion, tau, j_idx):
    """Expected reduction in Bayes decision risk if evidence source j is acquired."""
    p = _sigmoid(z)
    l1 = fusion.samples[(j_name, 1)]; l0 = fusion.samples[(j_name, 0)]
    w = fusion.w[j_idx]
    e1 = bayes_risk(_sigmoid(z + w * l1), tau).mean()
    e0 = bayes_risk(_sigmoid(z + w * l0), tau).mean()
    return float(bayes_risk(p, tau) - (p * e1 + (1 - p) * e0))

def entropy_gain(z, j_name, fusion, j_idx):
    def H(p): p = np.clip(p, 1e-9, 1 - 1e-9); return -(p * np.log(p) + (1 - p) * np.log(1 - p))
    p = _sigmoid(z); w = fusion.w[j_idx]
    e1 = H(_sigmoid(z + w * fusion.samples[(j_name, 1)])).mean()
    e0 = H(_sigmoid(z + w * fusion.samples[(j_name, 0)])).mean()
    return float(H(p) - (p * e1 + (1 - p) * e0))

# ---------------------------------------------------------------------
# Metrics
# ---------------------------------------------------------------------
def metrics(y, p):
    from sklearn.metrics import average_precision_score, roc_auc_score, brier_score_loss
    y = np.asarray(y); p = np.asarray(p)
    if y.sum() == 0 or y.sum() == len(y): return dict(AUPRC=np.nan, AUROC=np.nan, Brier=np.nan)
    return dict(AUPRC=average_precision_score(y, p), AUROC=roc_auc_score(y, p), Brier=brier_score_loss(y, np.clip(p, 0, 1)))

def cluster_bootstrap_diff(y, p_a, p_b, clusters, n_boot=1000, seed=0, metric="AUPRC"):
    """Paired cluster (product-code) bootstrap of metric(p_a) - metric(p_b)."""
    from sklearn.metrics import average_precision_score, roc_auc_score
    f = average_precision_score if metric == "AUPRC" else roc_auc_score
    rng = np.random.default_rng(seed)
    cl = np.asarray(clusters); u = np.unique(cl)
    idx = {c: np.where(cl == c)[0] for c in u}
    diffs = []
    for _ in range(n_boot):
        pick = np.concatenate([idx[c] for c in rng.choice(u, len(u), replace=True)])
        yy = y[pick]
        if yy.sum() == 0 or yy.sum() == len(yy): continue
        diffs.append(f(yy, p_a[pick]) - f(yy, p_b[pick]))
    d = np.array(diffs)
    return dict(diff=float(f(y, p_a) - f(y, p_b)), lo=float(np.percentile(d, 2.5)),
                hi=float(np.percentile(d, 97.5)), p_le0=float((d <= 0).mean()))

def cluster_bootstrap_ci(y, p, clusters, n_boot=1000, seed=0, metric="AUPRC"):
    from sklearn.metrics import average_precision_score, roc_auc_score
    f = average_precision_score if metric == "AUPRC" else roc_auc_score
    rng = np.random.default_rng(seed); cl = np.asarray(clusters); u = np.unique(cl)
    idx = {c: np.where(cl == c)[0] for c in u}; vals = []
    for _ in range(n_boot):
        pick = np.concatenate([idx[c] for c in rng.choice(u, len(u), replace=True)])
        if 0 < y[pick].sum() < len(pick): vals.append(f(y[pick], p[pick]))
    return float(np.percentile(vals, 2.5)), float(np.percentile(vals, 97.5))

def lead_time_table(cases, p, tau, events, horizon):
    """For every recall event in the evaluated period, earliest alert month before it."""
    df = cases[["code", "mi"]].copy(); df["p"] = p
    rows = []
    for code, e in events:
        w = df[(df.code == code) & (df.mi >= e - horizon) & (df.mi < e)]
        if len(w) == 0: continue
        hit = w[w.p >= tau]
        rows.append(dict(code=code, event_mi=e, detected=len(hit) > 0,
                         lead=(e - hit.mi.min()) if len(hit) else 0))
    return pd.DataFrame(rows)


# AEGIS-MD · Notebook 3 — Benchmark construction, routine detectors & gap-driven retrieval (CPU)
Inputs: attach the output of **Notebook 2**.

1. Builds the **leakage-free recall benchmark**: one case per (product code, month *t*); label = a new recall of the target class is initiated in *(t, t+6]*; months right after a recall are washed out (stimulated reporting); strict temporal train / validation / test split.
2. **Tier-0 routine surveillance evidence** (what standard post-market surveillance already sees): Poisson exceedance, CUSUM, Mann–Kendall trend & Sen slope, persistence, Isolation Forest anomaly, severity shift, problem-code distribution shift (Jensen–Shannon).
3. **Deep evidence** that requires *reading* reports, obtained through different retrieval strategies over the recent reports of that code:
   * **Gap-driven hybrid retrieval** (ours): one BM25 + dense query per evidence gap (mechanism, manufacturer investigation, batch/lot, recall precedent), conditioned on the signal's emerging problem terms;
   * generic-query RAG, random reading, and full reading (upper bound).

In [ ]:
import os, sys, json, time
import numpy as np, pandas as pd
sys.path.insert(0, os.getcwd())
from aegis_core import *
from collections import Counter
OUT = out_dir()

R = pd.read_pickle(find_file("records.pkl"))
ANN = pd.read_pickle(find_file("record_ann.pkl"))
E = np.load(find_file("rec_emb.npy"))
PROTO = pd.read_pickle(find_file("proto_emb.pkl"))
COUNTS = pd.read_pickle(find_file("counts.pkl"))
CODES = pd.read_pickle(find_file("codes.pkl"))
REC = pd.read_pickle(find_file("recalls.pkl"))
assert (ANN.rid.values == R.rid.values).all() and len(E) == len(R)
print(len(R), "records |", len(CODES), "codes |", len(REC), "recall rows")

MAND = R.source.str.lower().str.contains("manufacturer|user facility|distributor|importer").values
RA = dict(emb=E, has_text=(R.text_event.fillna("").str.len() > 20).values, mand_src=MAND,
          has_mfr=ANN.has_mfr.values.astype(bool), p_confirm=ANN.p_confirm.values, p_refute=ANN.p_refute.values,
          lot=R.lot.fillna("").replace({"UNK": "", "UNKNOWN": "", "NA": "", "N/A": "", "NI": ""}).values,
          model=R.model.fillna("").replace({"UNK": "", "UNKNOWN": "", "NA": "", "N/A": "", "NI": ""}).values,
          prec_sim=ANN.prec_sim.values.astype(np.float32))

## 1. Labels (recall ground truth) and case panel

In [ ]:
H, W = CFG["HORIZON"], CFG["WASHOUT"]
lab = REC[REC.klass.isin(CFG["LABEL_CLASSES"])]
ev = lab.groupby(["code", "event_id"]).init_mi.min().reset_index()
EVENTS = ev.groupby("code").init_mi.apply(lambda s: sorted(set(int(x) for x in s))).to_dict()
print("label classes:", CFG["LABEL_CLASSES"], "| recall events in panel codes:",
      sum(len(EVENTS.get(c, [])) for c in CODES.code))

R_SER = R.event_type.str.lower().isin(["death", "injury"]).values
agg = {}
for (c, m), g in R.groupby(["code", "mi"]):
    pc = Counter()
    for s in g.problems.fillna(""):
        for p in s.split(";"):
            p = p.strip()
            if p: pc[p] += 1
    ii = g.index.values
    agg[(c, m)] = dict(n=len(g), serious=int(R_SER[ii].sum()), mfr_src=int(MAND[ii].sum()),
                       text=int(RA["has_text"][ii].sum()), prob=pc)

cnt = COUNTS.set_index(["code", "mi"]).n.to_dict()
p0, p1 = ym(CFG["PANEL_START"]), ym(CFG["PANEL_END"])
first_m = ym(CFG["DATA_START"]) + CFG["RECENT"] + CFG["BASELINE"]
rows = []
for c in CODES.code:
    nb = {m: v for (cc, m), v in cnt.items() if cc == c}
    ag = {m: v for (cc, m), v in agg.items() if cc == c}
    t0 = tier0_for_code(nb, range(first_m, p1 + 1), ag)
    t0 = t0[t0.mi >= p0].copy(); t0["code"] = c
    evs = EVENTS.get(c, [])
    t0["y"] = [int(any(t < e <= t + H for e in evs)) for t in t0.mi]
    t0["washout"] = [int(any(t - W < e <= t for e in evs)) for t in t0.mi]
    rows.append(t0)
CASES = pd.concat(rows, ignore_index=True)
CASES = CASES[CASES.washout == 0].reset_index(drop=True)
CASES["split"] = [split_of(m) for m in CASES.mi]
CASES["case_id"] = np.arange(len(CASES))
print(CASES.groupby("split").y.agg(["count", "sum", "mean"]))

# Isolation Forest on routine multivariate surveillance features (fit on train only)
from sklearn.ensemble import IsolationForest
IF_FEATS = ["pois_score", "log_ratio", "mk_z", "sen_norm", "persist", "cusum", "sev_z", "prob_jsd"]
iso = IsolationForest(n_estimators=300, contamination="auto", random_state=CFG["SEED"]).fit(
    CASES.loc[CASES.split == "train", IF_FEATS].fillna(0))
CASES["if_score"] = -iso.score_samples(CASES[IF_FEATS].fillna(0))

## 2. Retrieval index (BM25 sparse + dense embeddings)

In [ ]:
t0 = time.time()
doc_text = (R.text_event.fillna("") + " " + R.text_mfr.fillna("") + " " + R.problems.fillna("").str.replace(";", " "))
BM = BM25Matrix(doc_text.values)
print("BM25 vocab", len(BM.vocab), f"{time.time() - t0:.0f}s")
BY = R.groupby(["code", "mi"]).rid.apply(np.array).to_dict()
names = CODES.set_index("code").device_name.to_dict()

def rows_in(c, lo, hi):
    parts = [BY[(c, m)] for m in range(lo, hi + 1) if (c, m) in BY]
    return np.concatenate(parts) if parts else np.array([], dtype=int)

## 3. Deep evidence under each retrieval strategy

In [ ]:
K = CFG["K_PER_GAP"]; KS = [10, 20, 40]
# corpus-level similarity thresholds (90th percentile) for the query-rate statistic
_samp = rng_thr = np.random.default_rng(0).choice(len(E), min(50000, len(E)), replace=False)
THR = {g: float(np.quantile(E[_samp].astype(np.float32) @ PROTO[g], 0.90)) for g in GAP_SPECS}
rng = np.random.default_rng(CFG["SEED"])
feat = {s: [] for s in ["full", "gap", "gap_nc"] + [f"generic{k}" for k in KS] + [f"random{k}" for k in KS]}
S_GAP, S_GAPB, S_GEN, CTX = {}, {}, {}, {}
t0 = time.time()
for i, r in CASES.iterrows():
    c, t = r.code, int(r.mi)
    win = rows_in(c, t - CFG["RECENT"] + 1, t)
    base = rows_in(c, t - CFG["RECENT"] - CFG["BASELINE"] + 1, t - CFG["RECENT"])
    base_all = base
    if len(base) > 400: base = rng.choice(base, 400, replace=False)
    if len(base):
        Eb = E[base].astype(np.float32); cb = Eb.mean(0); cb /= (np.linalg.norm(cb) + 1e-9)
        ctx = dict(base_cen=cb, base_coh=float((Eb @ cb).mean()))
    else:
        ctx = dict(base_cen=None, base_coh=0.0)
    wp, bp = Counter(), Counter()
    for m in range(t - CFG["RECENT"] + 1, t + 1): wp.update(agg.get((c, m), {}).get("prob", {}))
    for m in range(t - CFG["RECENT"] - CFG["BASELINE"] + 1, t - CFG["RECENT"] + 1): bp.update(agg.get((c, m), {}).get("prob", {}))
    em_phr, em_tok = emerging_terms(wp, bp)
    dn_tok = tokenize(names.get(c, ""))
    CTX[r.case_id] = dict(emerging=em_phr, n_window=len(win))

    f = deep_evidence(win, RA, ctx); f["n_docs"] = len(win); feat["full"].append(f)
    g_out, g_nc, s_lists, b_lists = {"n_window": len(win)}, {"n_window": len(win)}, {}, {}
    for gname, spec in GAP_SPECS.items():
        q = spec["terms"] + (em_tok + dn_tok if spec["use_emerging"] else [])
        S = hybrid_retrieve(win, BM, E, q, PROTO[gname], K)
        Sb = hybrid_retrieve(base, BM, E, q, PROTO[gname], K) if len(base) else np.array([], dtype=int)
        s_lists[gname] = S; b_lists[gname] = Sb
        g_nc.update(deep_evidence(S, RA, ctx, sources=[gname]))
        g_out.update(contrastive_gap_evidence(S, Sb, RA, ctx, gname,
                                              rate=query_rate_shift(win, base_all, E, PROTO[gname], THR[gname])))
    feat["gap"].append(g_out); feat["gap_nc"].append(g_nc); S_GAP[r.case_id] = s_lists; S_GAPB[r.case_id] = b_lists
    gq = GENERIC_QUERY["terms"] + dn_tok
    for k in KS:
        S = hybrid_retrieve(win, BM, E, gq, PROTO["generic"], k)
        if k == K: S_GEN[r.case_id] = S
        f = deep_evidence(S, RA, ctx); f["n_docs"] = len(S); feat[f"generic{k}"].append(f)
        S = rng.choice(win, min(k, len(win)), replace=False) if len(win) else win
        f = deep_evidence(S, RA, ctx); f["n_docs"] = len(S); feat[f"random{k}"].append(f)
    if i % 1000 == 0: print(i, "/", len(CASES), f"{time.time() - t0:.0f}s")

FEAT = {s: pd.DataFrame(v, index=CASES.case_id.values) for s, v in feat.items()}
print({s: v.shape for s, v in FEAT.items()})

In [ ]:
CASES.to_pickle(f"{OUT}/cases.pkl")
pd.to_pickle(FEAT, f"{OUT}/deep_features.pkl")
pd.to_pickle(dict(S_GAP=S_GAP, S_GAPB=S_GAPB, S_GEN=S_GEN, CTX=CTX), f"{OUT}/retrieval_sets.pkl")
pd.to_pickle(EVENTS, f"{OUT}/events.pkl")
summ = CASES.groupby("split").agg(cases=("y", "size"), positives=("y", "sum"), codes=("code", "nunique"))
summ.to_csv(f"{OUT}/benchmark_summary.csv"); print(summ)
# carry forward files needed by Notebooks 4-5
import shutil
for f in ["records.pkl", "recalls.pkl", "codes.pkl", "record_ann.pkl", "recall_texts.pkl", "enrich_meta.json", "data_summary.json"]:
    src = find_file(f, required=False)
    if src and os.path.abspath(src) != os.path.abspath(f"{OUT}/{f}"): shutil.copy(src, f"{OUT}/{f}")

In [ ]:
import gc; gc.collect(); print('section done')


# AEGIS-MD · Notebook 4 — Evidence fusion, Value-of-Information gap policy & full evaluation (CPU)
Inputs: attach the outputs of **Notebook 2** and **Notebook 3**.

* **Reliability-discounted Bayesian evidence fusion**: every evidence source is converted to a calibrated log-likelihood ratio (fit on train), discounted by the reliability of the documents it came from, and pooled with weights learned on validation. Contradictory evidence (e.g. a manufacturer investigation that could not confirm the failure) enters as negative log-odds.
* **Evidence-gap analysis as Value of Information (VoI)**: for each missing evidence type, the expected reduction in Bayes decision risk if it were retrieved. The investigation loop retrieves the highest-VoI gap, re-evaluates, and stops when no gap is worth its reading cost.
* Baselines: Poisson, CUSUM, Mann–Kendall, Isolation Forest, gradient boosting (routine and full features), fusion with generic-query RAG, random reading and full reading.
* Metrics on the held-out **test period**: AUPRC, AUROC, Brier, documents read, recall events detected and **lead time before recall initiation**, product-code cluster bootstrap CIs.

In [ ]:
import os, sys, json, time
import numpy as np, pandas as pd
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
sys.path.insert(0, os.getcwd())
from aegis_core import *
OUT = out_dir(); FIG = os.path.join(OUT, "figures"); os.makedirs(FIG, exist_ok=True)

CASES = pd.read_pickle(find_file("cases.pkl"))
FEAT = pd.read_pickle(find_file("deep_features.pkl"))
RS = pd.read_pickle(find_file("retrieval_sets.pkl"))
EVENTS = pd.read_pickle(find_file("events.pkl"))
y = CASES.y.values; split = CASES.split.values; code = CASES.code.values
tr, va, te = split == "train", split == "val", split == "test"
print(CASES.groupby("split").y.agg(["size", "sum"]))
ALERT_RATE = 0.10       # alert threshold = top 10% of validation risk scores
RESULTS, PRED = [], {}

def thr(p_val): return float(np.quantile(p_val, 1 - ALERT_RATE))
test_events = [(c, e) for c, evs in EVENTS.items() if c in set(code[te])
               for e in evs if e > ym(CFG["VAL_END"]) + 1 and e <= ym(CFG["PANEL_END"]) + CFG["HORIZON"]]
print("test recall events:", len(test_events))

def record(name, p_all, docs=0.0, family="", tau=None):
    m = metrics(y[te], p_all[te]); lo, hi = cluster_bootstrap_ci(y[te], p_all[te], code[te], n_boot=500)
    tau = thr(p_all[va]) if tau is None else tau
    lt = lead_time_table(CASES[te].reset_index(drop=True), p_all[te], tau, test_events, CFG["HORIZON"])
    det = lt.detected.mean() if len(lt) else np.nan
    lead = lt.loc[lt.detected, "lead"].median() if len(lt) and lt.detected.any() else np.nan
    RESULTS.append(dict(method=name, family=family, docs_read=round(float(docs), 2), AUPRC=m["AUPRC"],
                        AUPRC_lo=lo, AUPRC_hi=hi, AUROC=m["AUROC"], Brier=m["Brier"],
                        events_detected=det, median_lead_months=lead))
    PRED[name] = p_all
    print(f"{name:38s} docs={docs:6.1f}  AUPRC={m['AUPRC']:.3f} [{lo:.3f},{hi:.3f}]  AUROC={m['AUROC']:.3f}  detected={det:.2f}  lead={lead}")

## 1. Routine surveillance baselines (unsupervised scores)

In [ ]:
for f, nm in [("pois_score", "Poisson exceedance"), ("cusum", "Poisson CUSUM"), ("mk_z", "Mann-Kendall trend"),
              ("if_score", "Isolation Forest")]:
    s = CASES[f].fillna(0).values
    s = (s - s[tr].min()) / (s[tr].max() - s[tr].min() + 1e-9)
    record(nm, s, family="routine-unsupervised")

## 2. Supervised black-box baselines

In [ ]:
from sklearn.ensemble import HistGradientBoostingClassifier
T0F = sum(TIER0_GROUPS.values(), [])
DF = sum(DEEP_FEATURES.values(), [])
def gbm(X):
    m = HistGradientBoostingClassifier(max_iter=300, learning_rate=0.05, max_leaf_nodes=15, l2_regularization=1.0,
                                       early_stopping=True, validation_fraction=0.15, random_state=CFG["SEED"])
    m.fit(X[tr], y[tr]); return m.predict_proba(X)[:, 1]
X0 = CASES[T0F].values
Xfull = np.column_stack([X0, FEAT["full"][DF].values])
record("GBM (routine features)", gbm(X0), family="supervised")
record("GBM (routine + all deep, full reading)", gbm(Xfull), docs=FEAT["full"].n_docs[te].mean(), family="supervised")

## 3. Evidence fusion with different reading strategies

In [ ]:
G0 = dict(TIER0_GROUPS)
GA = {**TIER0_GROUPS, **DEEP_FEATURES}
def Xof(strategy):
    X = CASES.set_index("case_id")[T0F].copy()
    if strategy: X = X.join(FEAT[strategy])
    return X.reset_index(drop=True)

fz0 = EvidenceFusion(G0).fit(Xof(None)[tr], y[tr], Xof(None)[va], y[va], groups_tr=code[tr])
record("Fusion (routine evidence only)", fz0.predict(Xof(None)), family="fusion")

for s in ["random10", "random20", "random40", "generic10", "generic20", "generic40", "full"]:
    X = Xof(s); fz = EvidenceFusion(GA).fit(X[tr], y[tr], X[va], y[va], groups_tr=code[tr])
    lab = {"full": "Fusion + full reading (all recent reports)"}.get(s, f"Fusion + {'generic-query RAG' if s.startswith('generic') else 'random reading'} (k={s[-2:]})")
    record(lab, fz.predict(X), docs=FEAT[s].n_docs[te].mean(), family="fusion+reading")

## 4. AEGIS: gap-driven retrieval + VoI investigation loop

In [ ]:
XG = Xof("gap")
GC = {**TIER0_GROUPS, **CONTRAST_FEATURES}          # contrastive gap evidence + query-rate statistic
FZ = EvidenceFusion(GC).fit(XG[tr], y[tr], XG[va], y[va], groups_tr=code[tr])
SRC = list(GC.keys()); n0 = len(TIER0_GROUPS)
DEEP_IDX = {s: SRC.index(s) for s in DEEP_SOURCES}
CONTRIB = FZ.contributions(XG)                      # (n_cases, n_sources)  w * rho * LLR
Z0 = FZ.b0 + CONTRIB[:, :n0].sum(1)
TAU = thr(FZ.predict(XG)[va])
print("pooling weights:", dict(zip(SRC, np.round(FZ.w, 3))), "| tau =", round(TAU, 4))

def docs_read(cid, acquired):
    """Reports read = retrieved recent-window reports + retrieved baseline reports (both are read)."""
    if not acquired: return 0
    S, Sb = RS["S_GAP"][cid], RS.get("S_GAPB", {}).get(cid, {})
    w = set().union(*[set(S[g].tolist()) for g in acquired])
    b = set().union(*[set(np.asarray(Sb.get(g, [])).tolist()) for g in acquired]) if Sb else set()
    return len(w) + len(b)

CIDS = CASES.case_id.values
def run_policy(policy, eps=0.0, B=4, seed=0):
    rng = np.random.default_rng(seed)
    P = np.zeros(len(CASES)); D = np.zeros(len(CASES)); ACQ = []
    for i in range(len(CASES)):
        z = Z0[i]; acq = []
        for step in range(B):
            rem = [g for g in DEEP_SOURCES if g not in acq]
            if not rem: break
            if policy == "voi":
                v = {g: voi(z, g, FZ, TAU, DEEP_IDX[g]) for g in rem}
                g = max(v, key=v.get)
                if v[g] <= eps: break
            elif policy == "entropy":
                v = {g: entropy_gain(z, g, FZ, DEEP_IDX[g]) for g in rem}
                g = max(v, key=v.get)
                if v[g] <= eps: break
            elif policy == "fixed":
                g = rem[0]
            elif policy == "random":
                g = rem[rng.integers(len(rem))]
            acq.append(g); z += CONTRIB[i, DEEP_IDX[g]]
        P[i] = 1 / (1 + np.exp(-z)); D[i] = docs_read(CIDS[i], acq); ACQ.append(acq)
    return P, D, ACQ

curve = []
for B in [1, 2, 3, 4]:
    for pol in ["fixed", "random"]:
        P, D, _ = run_policy(pol, B=B)
        record(f"Gap retrieval, {pol} order (B={B})", P, docs=D[te].mean(), family="gap-ablation")
        curve.append(dict(policy=pol, param=B, docs=D[te].mean(), AUPRC=metrics(y[te], P[te])["AUPRC"]))
EPS = [0.0, 1e-5, 5e-5, 1e-4, 3e-4, 1e-3, 3e-3]
voi_runs = {}
for eps in EPS:
    for pol in ["voi", "entropy"]:
        e = eps if pol == "voi" else eps * 20
        P, D, A = run_policy(pol, eps=e)
        voi_runs[(pol, eps)] = (P, D, A)
        curve.append(dict(policy=pol, param=e, docs=D[te].mean(), AUPRC=metrics(y[te], P[te])["AUPRC"],
                          val_AUPRC=metrics(y[va], P[va])["AUPRC"], val_docs=D[va].mean()))
CURVE = pd.DataFrame(curve); CURVE.to_csv(f"{OUT}/efficiency_curve.csv", index=False)

# choose the operating point on VALIDATION only: among both VoI utilities (decision risk / information),
# the cheapest setting whose validation AUPRC is within 2% of the best validation AUPRC
cv = CURVE[CURVE.policy.isin(["voi", "entropy"])].copy()
ok = cv[cv.val_AUPRC >= 0.98 * cv.val_AUPRC.max()].sort_values("val_docs")
POL_STAR = ok.iloc[0].policy
EPS_STAR = [e for e in EPS if abs((e if POL_STAR == "voi" else e * 20) - ok.iloc[0].param) < 1e-12][0]
P_A, D_A, A_A = voi_runs[(POL_STAR, EPS_STAR)]
UTIL = {"voi": "decision-risk", "entropy": "information"}[POL_STAR]
record(f"AEGIS (VoI gap policy, {UTIL} utility)", P_A, docs=D_A[te].mean(), family="AEGIS")
for pol in ["voi", "entropy"]:
    if pol == POL_STAR: continue
    sub = CURVE[CURVE.policy == pol]; best_e = sub.sort_values("val_docs")[sub.val_AUPRC >= 0.98 * sub.val_AUPRC.max()].iloc[0].param
    e0 = [e for e in EPS if abs((e if pol == "voi" else e * 20) - best_e) < 1e-12][0]
    P_o, D_o, _ = voi_runs[(pol, e0)]
    record(f"Gap retrieval, VoI {'decision-risk' if pol == 'voi' else 'information'} utility (alt.)", P_o, docs=D_o[te].mean(), family="gap-ablation")
P_all, D_all, _ = run_policy("fixed", B=4)
record("AEGIS-all-gaps (every gap retrieved)", P_all, docs=D_all[te].mean(), family="AEGIS")

## 4b. AEGIS adaptive investigation: VoI decides *which signals* to investigate and *which evidence* to read
Evidence for an investigated case is extracted from the full set of recent reports of that product code;
cost = number of reports read. Compared with reading everything, triage by routine risk, and random triage.

In [ ]:
XF = Xof("full")
FZF = EvidenceFusion(GA).fit(XF[tr], y[tr], XF[va], y[va], groups_tr=code[tr])
SRCF = list(GA.keys()); DIDX = {s: SRCF.index(s) for s in DEEP_SOURCES}
CF = FZF.contributions(XF); Z0F = FZF.b0 + CF[:, :n0].sum(1)
NW = FEAT["full"].n_docs.fillna(0).values.astype(float)
TAUF = thr(FZF.predict(XF)[va])
P_FULL = FZF.predict(XF)

def run_gate(policy, eps=0.0, B=4):
    P = np.zeros(len(CASES)); D = np.zeros(len(CASES)); ACQ = []
    for i in range(len(CASES)):
        z = Z0F[i]; acq = []
        for _ in range(B):
            rem = [g for g in DEEP_SOURCES if g not in acq]
            if not rem: break
            if policy == "voi": v = {g: voi(z, g, FZF, TAUF, DIDX[g]) for g in rem}
            else: v = {g: entropy_gain(z, g, FZF, DIDX[g]) for g in rem}
            g = max(v, key=v.get)
            if v[g] <= eps: break
            acq.append(g); z += CF[i, DIDX[g]]
        P[i] = 1 / (1 + np.exp(-z)); D[i] = NW[i] if acq else 0.0; ACQ.append(acq)
    return P, D, ACQ

def triage(order_score, frac, seed=0):
    """Read (all deep evidence) for the top `frac` of cases ranked by order_score."""
    k = int(round(frac * len(CASES)))
    pick = np.zeros(len(CASES), bool)
    if k: pick[np.argsort(-order_score)[:k]] = True
    z = Z0F + np.where(pick, CF[:, n0:].sum(1), 0.0)
    return 1 / (1 + np.exp(-z)), np.where(pick, NW, 0.0)

gate_curve, gate_runs = [], {}
for pol, grid in [("voi", [0.0, 1e-5, 3e-5, 1e-4, 3e-4, 1e-3, 3e-3, 1e-2]), ("entropy", [0.0, 1e-4, 1e-3, 3e-3, 1e-2, 3e-2, 0.1, 0.2])]:
    for eps in grid:
        P, D, A = run_gate(pol, eps); gate_runs[(pol, eps)] = (P, D, A)
        gate_curve.append(dict(policy="AEGIS-" + pol, param=eps, docs=D[te].mean(), read_frac=(D[te] > 0).mean(),
                               AUPRC=metrics(y[te], P[te])["AUPRC"], AUROC=metrics(y[te], P[te])["AUROC"],
                               val_AUPRC=metrics(y[va], P[va])["AUPRC"], val_docs=D[va].mean()))
rng_t = np.random.default_rng(7)
for frac in [0.1, 0.2, 0.3, 0.4, 0.5, 0.7, 1.0]:
    for nm, sc in [("triage-by-routine-risk", Z0F), ("random-triage", rng_t.random(len(CASES)))]:
        P, D = triage(sc, frac)
        gate_curve.append(dict(policy=nm, param=frac, docs=D[te].mean(), read_frac=(D[te] > 0).mean(),
                               AUPRC=metrics(y[te], P[te])["AUPRC"], AUROC=metrics(y[te], P[te])["AUROC"],
                               val_AUPRC=metrics(y[va], P[va])["AUPRC"], val_docs=D[va].mean()))
GCURVE = pd.DataFrame(gate_curve); GCURVE.to_csv(f"{OUT}/adaptive_investigation_curve.csv", index=False)
print(GCURVE.round(4).to_string())

gv = GCURVE[GCURVE.policy.str.startswith("AEGIS")]
ok = gv[gv.val_AUPRC >= 0.98 * gv.val_AUPRC.max()].sort_values("val_docs").iloc[0]
pol_g = ok.policy.split("-")[1]; eps_g = ok.param
P_G, D_G, A_G = gate_runs[(pol_g, eps_g)]
record(f"AEGIS adaptive investigation (VoI-{pol_g})", P_G, docs=D_G[te].mean(), family="AEGIS-adaptive")
frac_match = float((D_G[te] > 0).mean())
for nm, sc in [("Triage by routine risk (matched reading)", Z0F), ("Random triage (matched reading)", np.random.default_rng(11).random(len(CASES)))]:
    P, D = triage(sc, frac_match)
    record(nm, P, docs=D[te].mean(), family="AEGIS-adaptive-baseline")
print(f"AEGIS adaptive: investigates {frac_match:.1%} of test product-code-months, reads {D_G[te].mean():.1f} reports/case vs {NW[te].mean():.1f} for full reading")

## 5. Ablations (component contribution)

In [ ]:
XNC = Xof("gap_nc")
fz_ncon = EvidenceFusion(GA).fit(XNC[tr], y[tr], XNC[va], y[va], groups_tr=code[tr])
record("Ablation: gap retrieval without contrast", fz_ncon.predict(XNC),
       docs=np.mean([len(set().union(*[set(RS["S_GAP"][c][g].tolist()) for g in DEEP_SOURCES])) for c in CIDS[te]]), family="ablation")
g_nrate = {k: [f for f in v if not f.startswith("rate_")] for k, v in GC.items()}
fz_nrate = EvidenceFusion(g_nrate).fit(XG[tr], y[tr], XG[va], y[va], groups_tr=code[tr])
record("Ablation: no query-rate statistic", fz_nrate.predict(XG), docs=D_all[te].mean(), family="ablation")
fz_nr = EvidenceFusion(GC, use_rho=False).fit(XG[tr], y[tr], XG[va], y[va], groups_tr=code[tr])
record("Ablation: no reliability discounting", fz_nr.predict(XG), docs=D_all[te].mean(), family="ablation")
g_nc = {k: [f for f in v if f != "inv_refute"] for k, v in GC.items()}
fz_nc = EvidenceFusion(g_nc).fit(XG[tr], y[tr], XG[va], y[va], groups_tr=code[tr])
record("Ablation: no contradictory evidence", fz_nc.predict(XG), docs=D_all[te].mean(), family="ablation")
for drop in DEEP_SOURCES:
    g = {k: v for k, v in GC.items() if k != drop}
    fz = EvidenceFusion(g).fit(XG[tr], y[tr], XG[va], y[va], groups_tr=code[tr])
    record(f"Ablation: without {drop} evidence", fz.predict(XG), docs=D_all[te].mean(), family="ablation")

## 6. Significance (paired product-code cluster bootstrap on the test period)

In [ ]:
RES = pd.DataFrame(RESULTS)
aegis = [r for r in RESULTS if r["family"] == "AEGIS-adaptive"][0]["method"]
aegis_gap = [r for r in RESULTS if r["family"] == "AEGIS"][0]["method"]
comps = [aegis_gap, "Triage by routine risk (matched reading)", "Random triage (matched reading)", "Fusion (routine evidence only)", "Fusion + generic-query RAG (k=20)", "Fusion + generic-query RAG (k=40)",
         "Fusion + random reading (k=20)", "GBM (routine features)", "GBM (routine + all deep, full reading)",
         "Fusion + full reading (all recent reports)", "Isolation Forest", "Poisson exceedance"]
sig = []
for cmp_ in comps:
    if cmp_ not in PRED: continue
    for met in ["AUPRC", "AUROC"]:
        d = cluster_bootstrap_diff(y[te], PRED[aegis][te], PRED[cmp_][te], code[te], n_boot=1000, metric=met)
        sig.append(dict(comparison=f"{aegis} vs {cmp_}", metric=met, **d))
SIG = pd.DataFrame(sig); SIG.to_csv(f"{OUT}/significance.csv", index=False)
print(SIG.round(4).to_string())

## 7. Does gap-driven retrieval find the *actual* future recall's failure mode? (root-cause alignment)

In [ ]:
REC = pd.read_pickle(find_file("recalls.pkl")); RT = pd.read_pickle(find_file("recall_texts.pkl"))
E = np.load(find_file("rec_emb.npy"), mmap_mode="r"); ER = np.load(find_file("recall_emb.npy"))
tidx = {t: i for i, t in enumerate(RT.text.values)}
REC["text"] = (REC.reason.fillna("") + " " + REC.root_cause.fillna("")).str.strip()
lab = REC[REC.klass.isin(CFG["LABEL_CLASSES"])]
rows = []
rng = np.random.default_rng(1)
for i in np.where(te & (y == 1))[0]:
    c, t, cid = code[i], int(CASES.mi.values[i]), CIDS[i]
    fut = lab[(lab.code == c) & (lab.init_mi > t) & (lab.init_mi <= t + CFG["HORIZON"])].text
    vid = [tidx[x] for x in fut if x in tidx]
    if not vid: continue
    Rv = ER[vid].astype(np.float32)
    def align(S):
        S = np.asarray(S, int)
        return float((E[S].astype(np.float32) @ Rv.T).max()) if len(S) else np.nan
    Sg, Sm = RS["S_GEN"].get(cid, []), RS["S_GAP"][cid]["mechanism"]
    win_n = RS["CTX"][cid]["n_window"]
    union = set()
    for g in DEEP_SOURCES: union |= set(RS["S_GAP"][cid][g].tolist())
    rows.append(dict(case_id=cid, gap_mechanism=align(Sm), generic=align(Sg),
                     gap_union=align(list(union)), n_window=win_n))
ALIGN = pd.DataFrame(rows)
if len(ALIGN):
    from scipy.stats import wilcoxon
    ok_ = ALIGN.dropna()
    w = wilcoxon(ok_.gap_mechanism, ok_.generic) if len(ok_) > 10 else None
    print(ALIGN[["gap_mechanism", "generic", "gap_union"]].describe().round(3))
    print("Wilcoxon gap(mechanism, k=10) vs generic (k=10):", w)
    ALIGN.to_csv(f"{OUT}/root_cause_alignment.csv", index=False)

## 8. Figures and tables

In [ ]:
RES = pd.DataFrame(RESULTS); RES.to_csv(f"{OUT}/results_test.csv", index=False)
print(RES[["method", "docs_read", "AUPRC", "AUPRC_lo", "AUPRC_hi", "AUROC", "Brier", "events_detected", "median_lead_months"]].round(3).to_string())

fig, ax = plt.subplots(figsize=(7.5, 5))
for pol, mk in [("voi", "o-"), ("entropy", "s--"), ("fixed", "^-"), ("random", "v:")]:
    d = CURVE[CURVE.policy == pol].sort_values("docs")
    ax.plot(d.docs, d.AUPRC, mk, label={"voi": "VoI, decision-risk utility", "entropy": "VoI, information utility",
                                         "fixed": "Gap retrieval, fixed order", "random": "Gap retrieval, random order"}[pol])
for s, mk in [("generic", "D"), ("random", "x")]:
    pts = RES[RES.method.str.contains("generic-query" if s == "generic" else "random reading")]
    ax.plot(pts.docs_read, pts.AUPRC, mk, ls="-", alpha=.7, label="Generic-query RAG" if s == "generic" else "Random reading")
for pol, mk in [("AEGIS-voi", "o-"), ("AEGIS-entropy", "s-"), ("triage-by-routine-risk", "^--"), ("random-triage", "x:")]:
    d = GCURVE[GCURVE.policy == pol].sort_values("docs")
    ax.plot(d.docs, d.AUPRC, mk, lw=2 if pol.startswith("AEGIS") else 1, label={"AEGIS-voi": "AEGIS adaptive (VoI decision-risk)", "AEGIS-entropy": "AEGIS adaptive (VoI information)", "triage-by-routine-risk": "Triage by routine risk", "random-triage": "Random triage"}[pol])
f0 = RES[RES.method == "Fusion (routine evidence only)"].AUPRC.values[0]
ff = RES[RES.method.str.startswith("Fusion + full")]
ax.axhline(f0, color="gray", ls=":", label="Routine evidence only")
ax.plot(ff.docs_read, ff.AUPRC, "*", ms=14, color="k", label="Full reading")
ax.set_xlabel("Mean documents read per product-code-month"); ax.set_ylabel("Test AUPRC")
ax.set_title("Evidence efficiency: detection quality vs reading cost"); ax.legend(fontsize=8); ax.grid(alpha=.3)
fig.tight_layout(); fig.savefig(f"{FIG}/fig_efficiency_curve.png", dpi=200)

fig, ax = plt.subplots(figsize=(7, 4))
main = RES[RES.family.isin(["routine-unsupervised", "supervised", "fusion", "AEGIS", "AEGIS-adaptive", "AEGIS-adaptive-baseline"]) | RES.method.str.contains("k=20|full reading")]
main = main.sort_values("AUPRC")
ax.barh(main.method, main.AUPRC, xerr=[main.AUPRC - main.AUPRC_lo, main.AUPRC_hi - main.AUPRC], color="#4C72B0")
ax.axvline(y[te].mean(), color="r", ls=":", label="prevalence"); ax.set_xlabel("Test AUPRC (95% cluster-bootstrap CI)")
ax.legend(); fig.tight_layout(); fig.savefig(f"{FIG}/fig_method_comparison.png", dpi=200)

acq = pd.Series([g for a, s in zip(A_G, te) if s for g in a]).value_counts()
n_gaps = pd.Series([len(a) for a, s in zip(A_G, te) if s]).value_counts().sort_index()
fig, axs = plt.subplots(1, 2, figsize=(9, 3.5))
acq.plot.bar(ax=axs[0], color="#55A868"); axs[0].set_title("Evidence gaps retrieved (test)")
n_gaps.plot.bar(ax=axs[1], color="#C44E52"); axs[1].set_title("Gaps retrieved per case"); axs[1].set_xlabel("# gaps")
fig.tight_layout(); fig.savefig(f"{FIG}/fig_gap_usage.png", dpi=200)

contrib = pd.DataFrame(np.abs(CF[te]), columns=SRCF).mean().sort_values()
fig, ax = plt.subplots(figsize=(6, 3.5)); contrib.plot.barh(ax=ax, color="#8172B2")
ax.set_xlabel("Mean |weighted log-odds contribution| (test)"); fig.tight_layout(); fig.savefig(f"{FIG}/fig_source_contributions.png", dpi=200)

# investigation trajectories (AEGIS adaptive) for NB5 explanations
traj = []
for i in np.where(te)[0]:
    z = Z0F[i]; steps = [dict(stage="routine", p=float(1 / (1 + np.exp(-z))))]
    for g in A_G[i]:
        z += CF[i, DIDX[g]]; steps.append(dict(stage=g, p=float(1 / (1 + np.exp(-z)))))
    traj.append(dict(case_id=int(CIDS[i]), code=code[i], mi=int(CASES.mi.values[i]), y=int(y[i]),
                     p_final=float(P_G[i]), acquired=A_G[i], steps=steps,
                     contrib={s: float(CF[i, j]) for j, s in enumerate(SRCF)}))
TAU = TAUF
pd.to_pickle(dict(traj=traj, tau=TAU, eps=eps_g, policy=pol_g, weights=dict(zip(SRCF, FZF.w.tolist())), b0=FZF.b0),
             f"{OUT}/aegis_test_trajectories.pkl")
PRED_DF = CASES[["case_id", "code", "mi", "split", "y"]].copy()
for k, v in PRED.items(): PRED_DF[k] = v
PRED_DF.to_pickle(f"{OUT}/predictions.pkl")
json.dump(dict(adaptive_policy=pol_g, adaptive_eps=eps_g, gap_policy=POL_STAR, eps_star=EPS_STAR, tau=TAU, alert_rate=ALERT_RATE, test_events=len(test_events),
               weights=dict(zip(SRCF, FZF.w.round(4).tolist()))), open(f"{OUT}/aegis_config.json", "w"), indent=1)
print("saved results, figures ->", OUT)

In [ ]:
import gc; gc.collect(); print('section done')


# AEGIS-MD · Notebook 5 — Evidence-grounded LLM explanations + faithfulness evaluation (GPU T4, Internet ON)
Inputs: attach outputs of **Notebook 2**, **Notebook 3** and **Notebook 4**.

For the top test-period alerts, an **evidence dossier** is assembled from the AEGIS investigation trajectory (routine signal → gaps retrieved → evidence for / against → posterior). An open LLM (`Qwen2.5-3B-Instruct`, fp16 on a T4) writes an investigation summary **that may only cite dossier items**. We then measure:
* **citation validity** (every cited ID exists in the dossier),
* **faithfulness** (NLI entailment of each cited sentence by its cited source),
* **root-cause anticipation**: similarity between the explanation and the text of the recall that was actually initiated later (true alerts only), comparing AEGIS dossiers with generic-RAG dossiers.

In [ ]:
import os, sys, json, re, time
import numpy as np, pandas as pd
sys.path.insert(0, os.getcwd())
from aegis_core import *
OUT = out_dir()
N_CASES = 40
LLM_NAME = "Qwen/Qwen2.5-3B-Instruct"      # or "Qwen/Qwen2.5-7B-Instruct" with LOAD_4BIT=True
LOAD_4BIT = False

T = pd.read_pickle(find_file("aegis_test_trajectories.pkl"))
CASES = pd.read_pickle(find_file("cases.pkl")).set_index("case_id")
R = pd.read_pickle(find_file("records.pkl")); ANN = pd.read_pickle(find_file("record_ann.pkl"))
REC = pd.read_pickle(find_file("recalls.pkl")); RS = pd.read_pickle(find_file("retrieval_sets.pkl"))
CODES = pd.read_pickle(find_file("codes.pkl")).set_index("code")
tau = T["tau"]
traj = sorted([t for t in T["traj"] if t["p_final"] >= tau], key=lambda d: -d["p_final"])
# keep at most 3 months per product code so the case set is diverse
seen, SEL = Counter(), []
for t in traj:
    if seen[t["code"]] < 3: SEL.append(t); seen[t["code"]] += 1
    if len(SEL) >= N_CASES: break
print("alerts at tau:", len(traj), "| selected:", len(SEL), "| true alerts among selected:", sum(t["y"] for t in SEL))

try:
    import torch; GPU = torch.cuda.is_available()
except Exception:
    torch = None; GPU = False
print("GPU:", GPU)

## 1. Evidence dossiers

In [ ]:
def snip(s, n=320):
    s = re.sub(r"\s+", " ", str(s or "")).strip(); return s[:n] + ("…" if len(s) > n else "")

def dossier(t, mode="aegis"):
    cid = t["case_id"]; c = CASES.loc[cid]; code = t["code"]
    items, lines = {}, []
    lines.append(f"Device product code {code} ({CODES.device_name.get(code, '')}); assessment month {ym_str(t['mi'])}.")
    lines.append(f"Routine surveillance: {int(c.n_recent)} reports in last 3 months vs baseline {c.base_mu * 3:.1f} expected "
                 f"(Poisson score {c.pois_score:.1f}); Mann-Kendall z={c.mk_z:.2f}; serious-event share {c.sev_share:.2f} "
                 f"(shift z={c.sev_z:.2f}); problem-code shift JSD={c.prob_jsd:.3f}.")
    if mode == "aegis":
        lines.append("Investigation trajectory: " + " -> ".join(f"{s['stage']} (risk {s['p']:.2f})" for s in t["steps"]))
        groups = {g: RS["S_GAP"][cid][g][:3] for g in t["acquired"]}
        if not groups: groups = {"mechanism": RS["S_GAP"][cid]["mechanism"][:3]}
    else:
        groups = {"retrieved": np.asarray(RS["S_GEN"].get(cid, []))[:6]}
    for g, S in groups.items():
        if g in GAP_SPECS: lines.append(f"Evidence gap '{g}': {GAP_SPECS[g]['question']}")
        for rid in S:
            r = R.iloc[int(rid)]; a = ANN.iloc[int(rid)]
            key = f"MDR:{r.mdr_key}"
            stance = ""
            if a.has_mfr and np.isfinite(a.p_confirm):
                stance = f" [manufacturer finding: confirmed={a.p_confirm:.2f}, not-confirmed={a.p_refute:.2f}]"
            txt = f"{snip(r.text_event, 260)} | Manufacturer: {snip(r.text_mfr, 220)} | lot {r.lot or 'n/a'}, model {r.model or 'n/a'}{stance}"
            items[key] = txt; lines.append(f"[{key}] {txt}")
            if g == "precedent" and a.prec_rec_id >= 0:
                pr = REC[REC.rec_id == a.prec_rec_id]
                if len(pr) and pr.posted_mi.iloc[0] < t["mi"]:
                    k2 = f"RECALL:{pr.recall_number.iloc[0] or pr.rec_id.iloc[0]}"
                    items[k2] = snip(pr.reason.iloc[0], 300)
                    lines.append(f"[{k2}] earlier recall (similarity {a.prec_sim:.2f}): {items[k2]}")
    if mode == "aegis":
        keep = set(TIER0_SOURCES) | set(t["acquired"])
        con = {k: v for k, v in t["contrib"].items() if abs(v) > 0.05 and k in keep}
        lines.append("Log-odds contributions (positive supports a safety signal, negative contradicts it): " +
                     ", ".join(f"{k} {v:+.2f}" for k, v in sorted(con.items(), key=lambda kv: -abs(kv[1]))))
        lines.append(f"Final AEGIS risk: {t['p_final']:.2f} (alert threshold {tau:.2f}).")
    return "\n".join(lines), items

PROMPT = """You are assisting a medical-device post-market safety reviewer. Using ONLY the evidence dossier below, write a concise investigation summary with these headings:
1. Signal  2. Supporting evidence  3. Contradicting or weakening evidence  4. Remaining evidence gaps  5. Suggested next step for the human reviewer.
Rules: cite every factual statement with the bracketed ID(s) from the dossier, e.g. [MDR:1234] or [RECALL:Z-0001-2020]. Do not invent facts, numbers or IDs. Do not claim a confirmed causal defect; this is a screening signal for human review. Max 220 words.

EVIDENCE DOSSIER:
{dossier}
"""

## 2. Generate explanations

In [ ]:
gen = None
if GPU:
    try:
        from transformers import AutoTokenizer, AutoModelForCausalLM
        tok = AutoTokenizer.from_pretrained(LLM_NAME)
        kw = dict(device_map="cuda", dtype=torch.float16)
        if LOAD_4BIT:
            from transformers import BitsAndBytesConfig
            kw["quantization_config"] = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16)
        llm = AutoModelForCausalLM.from_pretrained(LLM_NAME, **kw).eval()
        def gen(text):
            msgs = [{"role": "user", "content": PROMPT.format(dossier=text)}]
            prompt = tok.apply_chat_template(msgs, add_generation_prompt=True, tokenize=False)
            enc = tok(prompt, return_tensors="pt").to("cuda")
            with torch.no_grad():
                out = llm.generate(**enc, max_new_tokens=380, do_sample=False, pad_token_id=tok.eos_token_id)
            return tok.decode(out[0, enc["input_ids"].shape[1]:], skip_special_tokens=True).strip()
    except Exception as ex:
        print("LLM unavailable -> template explanations:", ex); gen = None

def template_expl(text, items):
    ids = list(items)[:4]
    return ("1. Signal: " + text.split("\n")[1] + "\n2. Supporting evidence: " + "; ".join(f"{items[i][:120]} [{i}]" for i in ids[:2]) +
            "\n3. Contradicting evidence: see manufacturer findings " + " ".join(f"[{i}]" for i in ids[2:4]) +
            "\n4. Remaining gaps: not all reports were read.\n5. Next step: human review of cited reports.")

EXPL = []
t0 = time.time()
for t in SEL:
    for mode in ["aegis", "generic"]:
        d, items = dossier(t, mode)
        e = gen(d) if gen else template_expl(d, items)
        EXPL.append(dict(case_id=t["case_id"], code=t["code"], mi=t["mi"], y=t["y"], p=t["p_final"], mode=mode,
                         dossier=d, items=items, explanation=e))
    print(len(EXPL), f"{time.time() - t0:.0f}s")

## 3. Faithfulness & root-cause anticipation

In [ ]:
CITE = re.compile(r"\[((?:MDR|RECALL):[^\]\s,;]+)\]")
nli = None
if GPU:
    try:
        from sentence_transformers import CrossEncoder
        nli = CrossEncoder("cross-encoder/nli-deberta-v3-base", device="cuda", max_length=384)
        ENT = {v.lower(): int(k) for k, v in nli.model.config.id2label.items()}["entailment"]
    except Exception as ex:
        print("NLI unavailable:", ex)
emb_model = None
if GPU:
    try:
        from sentence_transformers import SentenceTransformer
        emb_model = SentenceTransformer("BAAI/bge-small-en-v1.5", device="cuda")
    except Exception as ex:
        print("embedder unavailable:", ex)

lab = REC[REC.klass.isin(CFG["LABEL_CLASSES"])]
rows = []
for x in EXPL:
    sents = [s for s in re.split(r"(?<=[\.\!\?])\s+|\n", x["explanation"]) if len(s.strip()) > 15]
    cited = [(s, CITE.findall(s)) for s in sents]
    all_ids = [i for _, ids in cited for i in ids]
    valid = [i in x["items"] for i in all_ids]
    sup = []
    if nli is not None:
        pairs = [(x["items"][i], CITE.sub("", s)) for s, ids in cited for i in ids if i in x["items"]]
        if pairs:
            lg = nli.predict(pairs, apply_softmax=True); sup = list(lg[:, ENT] > 0.5)
    rc = np.nan
    if x["y"] == 1 and emb_model is not None:
        fut = lab[(lab.code == x["code"]) & (lab.init_mi > x["mi"]) & (lab.init_mi <= x["mi"] + CFG["HORIZON"])]
        if len(fut):
            v = emb_model.encode([x["explanation"]] + list((fut.reason + " " + fut.root_cause.fillna("")).values),
                                 normalize_embeddings=True)
            rc = float((v[1:] @ v[0]).max())
    rows.append(dict(case_id=x["case_id"], mode=x["mode"], y=x["y"], n_sent=len(sents),
                     cited_sent_frac=np.mean([bool(ids) for _, ids in cited]) if cited else 0,
                     n_citations=len(all_ids), citation_validity=np.mean(valid) if valid else np.nan,
                     nli_supported=np.mean(sup) if len(sup) else np.nan, rootcause_sim=rc))
FAITH = pd.DataFrame(rows)
print(FAITH.groupby("mode")[["cited_sent_frac", "citation_validity", "nli_supported", "rootcause_sim"]].mean().round(3))
if FAITH.rootcause_sim.notna().sum() > 10:
    from scipy.stats import wilcoxon
    pv = FAITH.pivot_table(index="case_id", columns="mode", values="rootcause_sim").dropna()
    if len(pv) > 5: print("root-cause anticipation, AEGIS vs generic dossier:", wilcoxon(pv.aegis, pv.generic))
FAITH.to_csv(f"{OUT}/explanation_faithfulness.csv", index=False)

with open(f"{OUT}/case_studies.md", "w") as f:
    for x in EXPL:
        if x["mode"] != "aegis": continue
        f.write(f"## {x['code']} · {ym_str(x['mi'])} · risk {x['p']:.2f} · {'TRUE recall followed' if x['y'] else 'no recall within 6 months'}\n\n")
        f.write("### Dossier\n```\n" + x["dossier"] + "\n```\n### Explanation\n" + x["explanation"] + "\n\n")
pd.to_pickle(EXPL, f"{OUT}/explanations.pkl")
json.dump(dict(llm=LLM_NAME if gen else "template", nli=bool(nli), n_cases=len(SEL)), open(f"{OUT}/explain_meta.json", "w"))
print("saved case_studies.md, explanation_faithfulness.csv")

In [ ]:
import gc; gc.collect(); print('section done')
